# Bit-flip rate vs cat size, with Krylov-Schur

Sweeps `alpha_sq` at fixed `eps_p` and compares against the perturbative
estimate, exactly as `notebooks/sweeps/legacy_cheb_ar/sweep_alpha.ipynb` does -- but each point is
solved with `KrylovSchurLindblad` (thick-restarted Arnoldi on the one-period
propagator, see `docs/krylov_schur.md`) instead of the Chebyshev-filtered
`ChebAr` pipeline, and the Hilbert-space truncation and the tolerances follow
a ladder in `alpha_sq` instead of being fixed along the sweep.

The points are independent (the interaction-frame basis does not move with
`alpha_sq`), so this fans out: **one job, one GPU task per point**, the
autoscaler sizing the pool. The coordinator function runs on a worker and asks
for no resources of its own -- each task requests its own GPU.

Unlike the `ChebAr` sweep there is no `floquet_lindblad.pipeline.solve_point` for this
solver, so the per-point work lives in this notebook as one picklable function
(`solve_point_ks`), in the shape of `run_krylov_schur` in
`notebooks/solvers/krylov_schur.ipynb`: plain numpy back, and nothing of size `N**2`
crosses the wire.

## What moves with `alpha_sq`

| `alpha_sq` | `n_a` / `n_b` | `m` / `k` | loop `rtol`/`atol` | final `rtol`/`atol` | `res_tol` | final block |
|---|---|---|---|---|---|---|
| `< 5`  | 15 / 7  | 40 / 10 | `1e-7` / `1e-8`  | `1e-9` / `1e-10`  | `1e-8`  | 10 |
| `< 7`  | 25 / 12 | 40 / 10 | `1e-8` / `1e-9`  | `1e-10` / `1e-11` | `1e-9`  | 20 |
| `< 10` | 35 / 16 | 50 / 12 | `1e-9` / `1e-10` | `1e-11` / `1e-12` | `1e-10` | 50 |
| `< 14` | 40 / 20 | 60 / 14 | `1e-10` / `1e-11`| `1e-12` / `1e-13` | `1e-11` | 100 |

The reasoning is in the schedule cell; the truncation check at the end says
whether the truncation column was generous enough at each point.

## Running it

1. **Kernel**: the workspace pixi environment,
   `/workspace/.pixi/envs/default/bin/python` (**Python 3.13.x**, `ray 2.56.0`)
   -- these must match the cluster.
2. **Vault session**, once (lasts about a week):

   ```bash
   export VAULT_ADDR=https://vault.int.alice-bob.com
   vault login -method=oidc role=research_core_user
   ```

3. Run all cells. The first `acr.run` of a session may open an interactive
   Microsoft device-code login for GCS read access; that is expected and cached.

If you have already run something that imports `ray` in this kernel, restart it
-- Ray caches `RAY_AUTH_MODE` at first import.

In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import numpy as np

from anb_compute import ray as acr

import pathlib


def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
PKG = str(REPO / "src" / "floquet_lindblad")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (PKG, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[PKG, DYNAMIQS])

print("ready")

### The `alpha_sq` ladder

Two things have to move together as the cat grows, and both are set here,
client-side, so the whole plan is printed before a single GPU is asked for.

In [ ]:
import numpy as np

EPS_P = 0.1
ALPHA_SQ_LIST = list(np.linspace(3, 14, 12))  # 3, 4, ..., 14

# --- what moves with alpha_sq ------------------------------------------------
#
# * The **truncation**. The memory holds <n_a> = alpha_sq / 2 photons in each
#   lobe and the buffer follows it, so a truncation that is comfortable at
#   alpha_sq = 3 leaks at alpha_sq = 14. The tail weight has to land far below
#   the residual we are claiming, which the last cell checks point by point.
#
# * The **tolerances**. What the iteration must resolve is `1 - mu` per period,
#   and that *is* the bit-flip rate: ~2.5e-6 at alpha_sq = 6 (`arnoldi`,
#   reports/benchmark_methods.md), falling roughly like exp(-alpha_sq) since
#   the perturbative rate goes as exp(-2|alpha|^2) and |alpha|^2 = alpha_sq / 2.
#   So, per band: ~2e-5 (alpha_sq = 4), 2.5e-6 (6), 3e-7 (8), 5e-8 (10),
#   ~1e-9 (14) -- the last one below any integrator tolerance we can afford on
#   a single period. Going up the ladder we therefore
#     - tighten the integrator for the polishing cycles and for the final
#       Rayleigh quotient (`rtol_final`), capped at 1e-12/1e-13, which is as
#       far as float64 goes without the step count exploding;
#     - lower `res_tol` with it, keeping it one decade above `rtol_final`
#       (the relative residual floors out at about the integrator tolerance,
#       so a smaller `res_tol` would just burn the cycle budget);
#     - lengthen the final block, so `1 - mu**n_blocks` stays ~1e5 x
#       `rtol_final` and the rate is read well above the noise:
#       1e-4 (band 1), 5e-5 (2), 2e-6 (3), 1e-7 (4);
#     - grow `m` and `k` a little: the slow cluster (target + phase-flip pair
#       + outer bulk) gets denser as the gap shrinks, and `k` must cover it.
#
# `max_cycles_loop` is left at the default (half of `max_cycles`): the relaxed
# phase has a residual floor of its own, so it is capped by cycles, not by
# `res_tol_loop`.

CALIBRATED_UP_TO = 14.0  # beyond this the last band is a guess, not a setting

SCHEDULE = [
    dict(alpha_sq_max=4.0, n_a=20, n_b=10, m=40, k=10,
         rtol_loop=1e-7, atol_loop=1e-8, res_tol_loop=1e-6,
         rtol_final=1e-9, atol_final=1e-10, res_tol=1e-8,
         n_blocks_final=10, max_cycles=25),
    dict(alpha_sq_max=7.0, n_a=25, n_b=12, m=40, k=10,
         rtol_loop=1e-8, atol_loop=1e-9, res_tol_loop=1e-7,
         rtol_final=1e-10, atol_final=1e-11, res_tol=1e-9,
         n_blocks_final=20, max_cycles=30),
    dict(alpha_sq_max=13, n_a=35, n_b=16, m=50, k=12,
         rtol_loop=1e-9, atol_loop=1e-10, res_tol_loop=1e-8,
         rtol_final=1e-11, atol_final=1e-12, res_tol=1e-10,
         n_blocks_final=50, max_cycles=35),
    dict(alpha_sq_max=np.inf, n_a=38, n_b=19, m=60, k=14,
         rtol_loop=1e-10, atol_loop=1e-11, res_tol_loop=1e-9,
         rtol_final=1e-12, atol_final=1e-13, res_tol=1e-11,
         n_blocks_final=100, max_cycles=40),
]


def point_settings(alpha_sq, schedule=SCHEDULE):
    """Truncation + tolerances for one point: the first band it falls in."""
    if alpha_sq > CALIBRATED_UP_TO:
        print(f"warning: alpha_sq = {alpha_sq:.2f} is past the calibrated "
              f"ladder (<= {CALIBRATED_UP_TO:g}); reusing its last band. Check "
              "the truncation tail and the residual by hand.")
    band = next(b for b in schedule if alpha_sq < b["alpha_sq_max"])
    return {k: v for k, v in band.items() if k != "alpha_sq_max"}


POINTS = [dict(alpha_sq=float(a), **point_settings(float(a))) for a in ALPHA_SQ_LIST]
COMMON = dict(eps_p=EPS_P, seed=0)

# The Krylov basis is (m+1) * (n_a*n_b)**2 complex numbers and lives on the
# GPU: 0.6 GB at the top of the ladder (40 x 20, m = 60), a few MB at the
# bottom. One GPU per point either way.
print(f"{'alpha_sq':>8} {'n_a':>4} {'n_b':>4} {'N':>5} {'m':>4} {'k':>3} "
      f"{'rtol_loop':>10} {'rtol_final':>11} {'res_tol':>8} {'blocks':>7} {'basis GB':>9}")
for p in POINTS:
    N = p["n_a"] * p["n_b"]
    print(f"{p['alpha_sq']:8.2f} {p['n_a']:4d} {p['n_b']:4d} {N:5d} {p['m']:4d} {p['k']:3d} "
          f"{p['rtol_loop']:10.0e} {p['rtol_final']:11.0e} {p['res_tol']:8.0e} "
          f"{p['n_blocks_final']:7d} {(p['m'] + 1) * N**2 * 16 / 1024**3:9.2f}")

### One point, on one GPU worker

In [ ]:
def solve_point_ks(
    *,
    alpha_sq,
    eps_p,
    n_a,
    n_b,
    # Krylov-Schur
    m=40,
    k=10,
    res_tol=1e-8,
    max_cycles=25,
    # integrator tolerance for the Arnoldi loop ...
    rtol_loop=1e-7,
    atol_loop=1e-8,
    # ... switched to the final tolerance once the residual reaches this, or
    # after `max_cycles_loop` cycles (the relaxed tolerance has a residual
    # floor of its own, which `res_tol_loop` may sit below) ...
    res_tol_loop=1e-6,
    max_cycles_loop=None,
    # ... which is also what the final Rayleigh quotient uses
    rtol_final=1e-9,
    atol_final=1e-10,
    n_blocks_final=10,
    seed=0,
    k_top=3,
):
    """One sweep point with `KrylovSchurLindblad`, on one GPU worker.

    Same two-phase schedule as `notebooks/solvers/krylov_schur.ipynb` (relaxed cycles,
    then polishing cycles at the final tolerance resuming the kept subspace,
    then a long-block Rayleigh quotient), but the payload is trimmed to what a
    sweep plots: no Krylov basis, no Ritz vector, no Wigner grid -- only the
    rates, the residuals, the cost, and the per-level Fock weights (n_a + n_b
    floats) that the truncation check needs.

    Physics constants come from `floquet_lindblad.models.ats` (single source of truth).
    """
    import time

    import numpy as np

    from floquet_lindblad import KrylovSchurLindblad
    from floquet_lindblad.models.ats import build_ats_hamiltonian_interaction

    def py(v):
        """0-d jax/numpy scalar -> plain python float/complex."""
        a = np.asarray(v)
        return complex(a) if np.iscomplexobj(a) else float(a)

    lines = []

    def log(s):
        print(s)
        lines.append(s)

    t0 = time.time()
    dims = (n_a, n_b)

    # --- interaction frame, ONE drive period per block: the iteration runs on U ---
    (
        H_I,
        jump_ops_I,
        jump_ops_LdL_I,
        output_phase,
        V,
        T_block,
        params,
    ) = build_ats_hamiltonian_interaction(
        n_a=n_a,
        n_b=n_b,
        alpha_sq=alpha_sq,
        epsilon_p=eps_p,
        n_periods=1,
    )

    solver = KrylovSchurLindblad(
        H_I,
        jump_ops_I,
        T_block,
        jump_ops_LdL=jump_ops_LdL_I,
        dims=dims,
        output_phase=output_phase,
        rtol=rtol_loop,
        atol=atol_loop,
    )
    x0 = solver.make_x0(seed=seed)
    t_build = time.time() - t0

    # --- phase 1: cycles at the loop tolerance -------------------------------
    same_tol = (rtol_loop, atol_loop) == (rtol_final, atol_final)
    if max_cycles_loop is None:
        max_cycles_loop = max(1, max_cycles // 2)
    t1 = time.time()
    out = solver.solve(
        x0,
        m=m,
        k=k,
        res_tol=res_tol if same_tol else res_tol_loop,
        max_cycles=max_cycles if same_tol else max_cycles_loop,
        log=log,
    )
    history = list(out["history"])
    n_cycles_loop = out["n_cycles"]
    t_loop = time.time() - t1

    # --- phase 2: polish at the final tolerance, resuming the kept subspace ---
    t2 = time.time()
    if not same_tol:
        log(f"-- switching integrator tolerance to rtol={rtol_final:g}, atol={atol_final:g}")
        solver.set_tolerance(rtol_final, atol_final)
        out = solver.solve(
            state=out["state"],
            m=m,
            k=k,
            res_tol=res_tol,
            max_cycles=max(1, max_cycles - n_cycles_loop),
            log=log,
        )
        history += list(out["history"])
    t_polish = time.time() - t2

    x_ritz = out["x_ritz"]

    # --- final numbers on the true propagator, tight tolerance ---------------
    # One block: residual of the eigenvector. n_blocks_final blocks: the
    # Rayleigh quotient that resolves 1 - mu**n above the integrator tolerance;
    # `rate_RQ` is the rate this sweep plots.
    t3 = time.time()
    res_1 = solver.residual_check(x_ritz, n_blocks=1, rtol=rtol_final, atol=atol_final)
    res_n = solver.residual_check(
        x_ritz, n_blocks=n_blocks_final, rtol=rtol_final, atol=atol_final
    )
    t_final = time.time() - t3

    # --- Fock weights, for the truncation check (lab frame, as in the other
    # notebooks). Only the two marginals travel, not rho itself.
    V = np.asarray(V)
    rho_lab = V @ np.asarray(solver.unvec(x_ritz)) @ V.conj().T
    D = np.einsum("ij,ij->i", rho_lab, rho_lab.conj()).real.reshape(n_a, n_b)
    w_a = D.sum(axis=1) / D.sum()
    w_b = D.sum(axis=0) / D.sum()

    import jax

    return {
        # sweep coordinates
        "alpha_sq": float(alpha_sq),
        "eps_p": float(eps_p),
        "n_a": int(n_a),
        "n_b": int(n_b),
        # eigenvalue from the Krylov iteration (one period) ...
        "mu": complex(out["mu"]),
        "rate_ritz": py(solver.rate_from_mu(out["mu"])),
        "res_rel": float(out["res_rel"]),
        "converged": bool(out["converged"]),
        # ... and from the final Rayleigh quotients at tight tolerance
        "mu_RQ_1": py(res_1["mu_RQ"]),
        "res_rel_1": py(res_1["res_rel"]),
        "mu_RQ_n": py(res_n["mu_RQ"]),
        "rate_bf": py(res_n["rate_RQ"]),  # named as in `pipeline.solve_point`
        "res_rel_n": py(res_n["res_rel"]),
        "n_blocks_final": int(n_blocks_final),
        # cost
        "n_apply": int(out["n_apply"]),
        "n_cycles": len(history),
        "n_cycles_loop": int(n_cycles_loop),
        "res_hist": np.array([h["res_rel"] for h in history]),
        "apply_hist": np.array([h["n_apply"] for h in history]),
        "mu_hist": np.array([h["mu"] for h in history]),
        "log": lines,
        # truncation check
        "fock_weight_a": w_a,
        "fock_weight_b": w_b,
        "tail_a": float(w_a[-k_top:].sum()),
        "tail_b": float(w_b[-k_top:].sum()),
        "k_top": int(k_top),
        "params": {kk: py(v) for kk, v in params.items()},
        "T_block": py(T_block),
        "settings": {
            "n_a": n_a, "n_b": n_b, "m": m, "k": k,
            "res_tol": res_tol, "res_tol_loop": res_tol_loop,
            "max_cycles": max_cycles, "max_cycles_loop": max_cycles_loop,
            "rtol_loop": rtol_loop, "atol_loop": atol_loop,
            "rtol_final": rtol_final, "atol_final": atol_final,
            "n_blocks_final": n_blocks_final, "seed": seed,
        },
        "timings_s": {
            "build": t_build,
            "loop": t_loop,
            "polish": t_polish,
            "final_RQ": t_final,
            "total": time.time() - t0,
        },
        "device": str(jax.devices()[0]),
    }


#: Coordinates copied onto a failed point, so it still says where it was.
_COORD_KEYS = ("alpha_sq", "eps_p", "n_a", "n_b", "m", "k", "res_tol")


def solve_point_ks_safe(**kwargs):
    """`solve_point_ks`, returning a failure instead of raising it.

    Same contract as `floquet_lindblad.pipeline.solve_point_safe`: one unconvergeable
    point is recorded and the sweep carries on, and a raising task does not
    abort the driver's `ray.wait` loop and discard the points already done.
    """
    try:
        return solve_point_ks(**kwargs)
    except Exception as exc:  # noqa: BLE001 - one bad point must not sink the sweep
        entry = {k: kwargs.get(k) for k in _COORD_KEYS}
        entry["error"] = f"{type(exc).__name__}: {exc}"
        return entry

### The sweep

In [ ]:
import time


def sweep(worker, points, common):
    """Coordinator: fans one GPU task out per point, gathers as they land.

    Runs as the driver on a worker (acr.run places it there because it is called
    with no resources of its own); each task asks for its own GPU. `worker` is
    passed in rather than imported: it is defined in this notebook, so it ships
    by value with this function.
    """
    import ray

    from anb_compute import ray as acr

    refs = {
        acr.submit(worker, num_gpus=1, num_cpus=8, **common, **p): p["alpha_sq"]
        for p in points
    }

    out = []
    pending = list(refs)
    while pending:
        ready, pending = ray.wait(pending, num_returns=1)
        ref, alpha_sq = ready[0], refs[ready[0]]
        try:
            res = ray.get(ref)
        except Exception as exc:  # a task lost to infra, not to numerics
            out.append({"alpha_sq": alpha_sq,
                        "error": f"{type(exc).__name__}: {exc}"})
            continue
        out.append(res)
        print(f"alpha_sq = {alpha_sq:5.2f}  "
              + (res["error"] if "error" in res
                 else f"rate_bf = {res['rate_bf'].real:.6e}  "
                      f"res = {res['res_rel']:.1e}  "
                      f"apps = {res['n_apply']:d}"
                      + ("" if res["converged"] else "  (NOT converged)")))
    out.sort(key=lambda d: d["alpha_sq"])
    return out


t0 = time.time()
results = acr.run(
    sweep, solve_point_ks_safe, POINTS, COMMON,
    runtime_env=RUNTIME_ENV,
    poll_seconds=5,
    # No resources here on purpose: the fan-out inside asks for its own.
)
print(f"\nwall clock, incl. job startup: {time.time() - t0:.1f} s "
      f"for {len(results)} point(s)")

### Results

In [ ]:
ok = [r for r in results if "error" not in r]
bad = [r for r in results if "error" in r]

alpha_sq = np.array([r["alpha_sq"] for r in ok])
bf_list = np.array([r["rate_bf"] for r in ok])
res_rel = np.array([r["res_rel_n"] for r in ok])

print(f"{'alpha_sq':>9} {'n_a x n_b':>10} {'rate_bf':>14} {'1-Re mu':>10} "
      f"{'res Krylov':>11} {'res RQ':>9} {'blocks':>7} {'apps':>6} {'cyc':>4} "
      f"{'conv':>5} {'GPU s':>7}")
for r in ok:
    print(f"{r['alpha_sq']:9.2f} {r['n_a']:4d} x{r['n_b']:4d} "
          f"{r['rate_bf'].real:14.6e} {1 - r['mu'].real:10.2e} "
          f"{r['res_rel']:11.2e} {r['res_rel_n']:9.2e} {r['n_blocks_final']:7d} "
          f"{r['n_apply']:6d} {r['n_cycles']:4d} "
          f"{('yes' if r['converged'] else 'NO'):>5} "
          f"{r['timings_s']['loop'] + r['timings_s']['polish']:7.1f}")
for r in bad:
    print(f"{r['alpha_sq']:9.2f}  FAILED: {r['error']}")

stalled = [r for r in ok if not r["converged"]]
if stalled:
    print("\nnot converged (residual above res_tol at the cycle budget): "
          + ", ".join(f"alpha_sq={r['alpha_sq']:.2f} (res {r['res_rel']:.1e} "
                      f"> {r['settings']['res_tol']:.0e})" for r in stalled))

### Bit-flip rate vs cat size

In [ ]:
import matplotlib.pyplot as plt

# kappa_1 / kappa_2 come back in each point's `params`, derived on the worker
# from the constants in floquet_lindblad.models.ats -- so they stay the single source of
# truth and are not re-derived here (this client cannot even import ats.py,
# which needs jax + dynamiqs).
kappa_1 = np.array([r["params"]["kappa_1"] for r in ok])
kappa_2 = np.array([r["params"]["kappa_2"] for r in ok])

# Perturbative estimate, kept verbatim from the pre-cluster notebook, including
# the x3000 eyeball normalisation -- it is a shape comparison, not a calibrated
# prefactor. Note `alpha_sq / 2` is what the axis calls |alpha|^2; the naming in
# the model builder is the other convention.
n = alpha_sq / 2
pert_th = (
    kappa_1 * n * np.exp(-4 * n)
    + kappa_1**2 / (2 * kappa_2) * np.exp(-2 * n)
)

converged = np.array([r["converged"] for r in ok])

fig, ax = plt.subplots(figsize=(5.5, 4.0), constrained_layout=True)
ax.scatter(n[converged], bf_list.real[converged], label="Krylov-Schur", zorder=3)
if (~converged).any():
    ax.scatter(n[~converged], bf_list.real[~converged], facecolors="none",
               edgecolors="#eb6834", zorder=3, label="not converged")
ax.plot(n, pert_th * 3000, ls=":", c="k", label=r"perturbative $\times$ 3000")
ax.set_yscale("log")
ax.set_xlabel(r"$|\alpha|^2$")
ax.set_ylabel("bit-flip rate (GHz)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### Truncation check

The weight in the top `k_top` levels of each mode has to sit far below the
residual we are quoting -- otherwise the truncation, not the solver, is what
sets the number. This is the same criterion as in
`notebooks/solvers/arnoldi_no_cheb.ipynb` and `notebooks/solvers/krylov_schur.ipynb`, run at
every point of the ladder: it is what says whether a band's `n_a` / `n_b` was
generous enough.

In [ ]:
TOL = 1e-11  # same absolute criterion as the single-point notebooks

print(f"{'alpha_sq':>9} {'n_a':>4} {'n_b':>4} {'tail_a':>10} {'tail_b':>10} "
      f"{'res RQ':>10} {'1-|mu|':>10} {'verdict':>8}")
worst = []
for r in ok:
    k_top = r["k_top"]
    tail_a, tail_b = r["tail_a"], r["tail_b"]
    passed = max(tail_a, tail_b) < TOL
    if not passed:
        worst.append(r)
    print(f"{r['alpha_sq']:9.2f} {r['n_a']:4d} {r['n_b']:4d} "
          f"{tail_a:10.2e} {tail_b:10.2e} {r['res_rel_n']:10.2e} "
          f"{1 - abs(r['mu']):10.2e} {('PASS' if passed else 'FAIL'):>8}")
print(f"\ntop-{ok[0]['k_top']} weight against tol = {TOL:.0e}: "
      + ("all points PASS" if not worst else
         "raise n_a / n_b for alpha_sq = "
         + ", ".join(f"{r['alpha_sq']:.2f}" for r in worst)))

# Per-level tails of the two extreme points, to see how much room a band has.
for r in (ok[0], ok[-1]):
    print(f"\nalpha_sq = {r['alpha_sq']:.2f}  (n_a = {r['n_a']}, n_b = {r['n_b']})")
    for label, w in (("a (storage)", r["fock_weight_a"]), ("b (buffer)", r["fock_weight_b"])):
        top = " ".join(f"{v:.1e}" for v in np.asarray(w)[-4:])
        print(f"  mode {label:<12} last 4 weights: {top}")

### Convergence along the ladder

One line per point: the target residual against period integrations. The curves
should step down cycle by cycle and flatten only at the point's `res_tol`; a
curve that flattens well above it is the integrator floor of that band, i.e. a
`rtol_final` that needs tightening (or a `res_tol` that was too ambitious).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.9), constrained_layout=True)

cmap = mpl.colormaps["viridis"]
norm = mpl.colors.Normalize(vmin=alpha_sq.min(), vmax=alpha_sq.max())
for r in ok:
    c = cmap(norm(r["alpha_sq"]))
    axes[0].plot(r["apply_hist"], r["res_hist"], ".-", ms=4, lw=1.1, color=c)
    axes[0].plot(r["apply_hist"][-1], r["settings"]["res_tol"], "_", ms=8, color=c)

axes[0].set_yscale("log")
axes[0].set_xlabel("period integrations")
axes[0].set_ylabel(r"residual $\|U y - \theta y\| / |\theta|$")
axes[0].set_title("target residual per cycle (tick: that point's res_tol)", fontsize=9)
fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes[0],
             label=r"$\alpha_{sq}$")

# Cost of a point, split between the two tolerance phases.
apps = np.array([r["n_apply"] for r in ok])
t_loop = np.array([r["timings_s"]["loop"] for r in ok])
t_pol = np.array([r["timings_s"]["polish"] for r in ok])
t_rq = np.array([r["timings_s"]["final_RQ"] for r in ok])
w = 0.6 * np.min(np.diff(np.sort(alpha_sq))) if len(alpha_sq) > 1 else 0.6
axes[1].bar(alpha_sq, t_loop, width=w, label="loop (relaxed)", color="#2a78d6")
axes[1].bar(alpha_sq, t_pol, width=w, bottom=t_loop, label="polish (tight)",
            color="#eb6834")
axes[1].bar(alpha_sq, t_rq, width=w, bottom=t_loop + t_pol,
            label="final Rayleigh quotient", color="#8c8c8c")
for x, y, a_ in zip(alpha_sq, t_loop + t_pol + t_rq, apps):
    axes[1].annotate(f"{a_}", (x, y), textcoords="offset points", xytext=(0, 2),
                     ha="center", fontsize=6, color="0.3")
axes[1].set_xlabel(r"$\alpha_{sq}$")
axes[1].set_ylabel("GPU time (s)")
axes[1].set_title("cost per point (label: period integrations in the loop)", fontsize=9)
axes[1].legend(fontsize=8, frameon=False)

for ax in axes:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, which="major", lw=0.4, color="0.9")
    ax.set_axisbelow(True)
plt.show()